# Task Arc

In [1]:
import os
import json
from datasets import load_dataset
from datasets import get_dataset_config_names
from datasets import get_dataset_split_names

In [2]:
def print_colored(convo, limit=float('inf')):
    for i, message in enumerate(convo['messages']):
        if i >= limit:
            print(f"\033[31m... {len(convo['messages']) - limit} more messages ...\033[0m")
            break
        role = message['role']
        content = message['content']
        if role == 'system':
            print(f"\033[33m{content}\033[0m")  # yellow
        elif role == 'assistant':
            print(f"\033[34m{content}\033[0m")  # blue
        elif role == 'user':
            print(f"\033[32m{content}\033[0m")  # green
        else:
            print(f"\033[31m{content}\033[0m")  # red

In [3]:
subsets = get_dataset_config_names("allenai/ai2_arc")
print(f"Subsets: {subsets}")


splits = get_dataset_split_names("allenai/ai2_arc", "ARC-Easy")
print(f"Splits: {splits}")
splits = get_dataset_split_names("allenai/ai2_arc", "ARC-Challenge")
print(f"Splits: {splits}")

Subsets: ['ARC-Challenge', 'ARC-Easy']
Splits: ['train', 'test', 'validation']
Splits: ['train', 'test', 'validation']


In [4]:
ds = load_dataset("allenai/ai2_arc", "ARC-Easy", split="train")

In [5]:
example = ds[0]
print(f"Example: {json.dumps(example, indent=2)}")

Example: {
  "id": "Mercury_7220990",
  "question": "Which factor will most likely cause a person to develop a fever?",
  "choices": {
    "text": [
      "a leg muscle relaxing after exercise",
      "a bacterial population in the bloodstream",
      "several viral particles on the skin",
      "carbohydrates being digested in the stomach"
    ],
    "label": [
      "A",
      "B",
      "C",
      "D"
    ]
  },
  "answerKey": "B"
}


In [6]:
question = example["question"]
choices = example["choices"]["text"]  # list of str
letters = example["choices"]["label"]  # e.g. ["A", "B", "C", "D"]
answer = example["answerKey"]  # e.g. "A"
assert answer in letters

# Same format as MMLU - note letter at the end and no space before letter (both better for small LLM)
user_message = f"Multiple Choice question: {question}\n"
user_message += "".join([f"- {choice}={letter}\n" for letter, choice in zip(letters, choices)])
user_message += "\nRespond only with the letter of the correct answer."
agent_message = f"{answer}"

convo = {
    "messages": [
        {"role": "user", "content": user_message},
        {"role": "assistant", "content": agent_message},
    ],
    "eval": {
        "letters": letters,  # used to focus logits during eval
        "answer": answer
    }
}

In [7]:
print(f"Convo: {json.dumps(convo, indent=2)}")

Convo: {
  "messages": [
    {
      "role": "user",
      "content": "Multiple Choice question: Which factor will most likely cause a person to develop a fever?\n- a leg muscle relaxing after exercise=A\n- a bacterial population in the bloodstream=B\n- several viral particles on the skin=C\n- carbohydrates being digested in the stomach=D\n\nRespond only with the letter of the correct answer."
    },
    {
      "role": "assistant",
      "content": "B"
    }
  ],
  "eval": {
    "letters": [
      "A",
      "B",
      "C",
      "D"
    ],
    "answer": "B"
  }
}


In [8]:
print_colored(convo)

Multiple Choice question: Which factor will most likely cause a person to develop a fever?
- a leg muscle relaxing after exercise=A
- a bacterial population in the bloodstream=B
- several viral particles on the skin=C
- carbohydrates being digested in the stomach=D

Respond only with the letter of the correct answer.
B


In [10]:
class TaskArc:
    def __init__(self, subset, split, stop=None):
        self.dataset = load_dataset("allenai/ai2_arc", subset, split=split)
        self.dataset = self.dataset.shuffle(seed=42)
        self.length = stop if stop is not None else len(self.dataset)
    
    def __len__(self):
        return self.length
    
    def __getitem__(self, idx):
        if idx >= self.length:
            raise IndexError(idx)
        example = self.dataset[idx]

        question = example["question"]
        choices = example["choices"]["text"]  # list of str
        letters = example["choices"]["label"]  # e.g. ["A", "B", "C", "D"]
        answer = example["answerKey"]  # e.g. "A"
        assert answer in letters

        # Same format as MMLU - note letter at the end and no space before letter (both better for small LLM)
        user_message = f"Multiple Choice question: {question}\n"
        user_message += "".join([f"- {choice}={letter}\n" for letter, choice in zip(letters, choices)])
        user_message += "\nRespond only with the letter of the correct answer."
        agent_message = f"{answer}"

        convo = {
            "messages": [
                {"role": "user", "content": user_message},
                {"role": "assistant", "content": agent_message},
            ],
            "eval": {
                "letters": letters,  # used to focus logits during eval
                "answer": answer
            }
        }
        return convo

    def evaluate(self, assistant_response, eval_data):
        assert isinstance(assistant_response, str)
        return assistant_response == eval_data["answer"]

In [13]:
def check_schema(convo):
    assert isinstance(convo, dict)
    assert convo.keys() == {'messages', 'eval'}
    assert isinstance(convo['messages'], list)
    for message in convo['messages']:
        assert isinstance(message, dict)
        assert message.keys() == {'role', 'content'}
        assert message['role'] in {'system', 'assistant', 'user'}
        assert isinstance(message['content'], str)
        assert len(message['content']) > 0
    assert isinstance(convo['eval'], dict)
    assert 'letters' in convo['eval']
    assert 'answer' in convo['eval']

In [ ]:
# Checked ok
for split in ["train", "validation", "test"]:
    task = TaskArc("ARC-Challenge", split)
    for i, e in enumerate(task):
        check_schema(e)
        assistant_response = e['messages'][-1]['content']
        assert task.evaluate(assistant_response, e['eval'])
        assert not task.evaluate("X", e['eval'])  # wrong answer
        if i % 10_000 == 0:
            print(f"Checked {i} / {len(task)} examples...")

Checked 0 / 1119 examples...
Checked 0 / 299 examples...
Checked 0 / 1172 examples...


In [ ]:
# Checked ok
for split in ["train", "validation", "test"]:
    task = TaskArc("ARC-Easy", split)
    for i, e in enumerate(task):
        check_schema(e)
        assistant_response = e['messages'][-1]['content']
        assert task.evaluate(assistant_response, e['eval'])
        assert not task.evaluate("X", e['eval'])  # wrong answer
        if i % 10_000 == 0:
            print(f"Checked {i} / {len(task)} examples...")

Checked 0 / 2251 examples...
Checked 0 / 570 examples...
Checked 0 / 2376 examples...
